In [1]:
import nest_asyncio

nest_asyncio.apply()

In [2]:
import os

from dotenv import load_dotenv

load_dotenv()

False

In [ ]:
# colab-only
!pip install giskard-checks langgraph langchain langchain-openai

The previous tutorials tested plain functions and single LLM calls. A real
agent adds a third thing to get wrong: the trajectory — which tools it decided
to call, in what order, and with which arguments. An agent can produce a
perfectly plausible answer while never having called the tool that would have
made it true.

## What you'll build

By the end of this tutorial you will have:

1. A minimal LangGraph ReAct agent with a `get_weather` and a
   `convert_currency` tool
2. Checks on the agent's final answer
3. Checks on its trajectory — that it called the right tool with the right
   arguments, and that it called *no* tool when none was needed
4. A `WithSpy` check that asserts on the arguments an internal call received
5. A `Suite` that runs all of it in one `await`

## Prerequisites

- Completed [Test Suites](/oss/checks/tutorials/test-suites)
- An OpenAI API key set in `OPENAI_API_KEY`
- `pip install giskard-checks langgraph langchain langchain-openai`

## 1. Build the agent

Two tools, each delegating to a plain module-level function. That indirection
is deliberate: in step 5 we patch `weather_api` to inspect the arguments it
received, and patching a plain function is simpler than patching a tool object
that the compiled graph already holds a reference to.

In [4]:
def weather_api(city: str) -> str:
    """Stand-in for a real weather HTTP call."""
    return f"{city}: 18C, light rain"


def fx_rate(from_currency: str, to_currency: str) -> float:
    """Stand-in for a real FX rate lookup."""
    return {("EUR", "USD"): 1.09, ("USD", "EUR"): 0.92}.get(
        (from_currency, to_currency), 1.0
    )

In [5]:
from langchain_core.tools import tool


@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    return str(weather_api(city))


@tool
def convert_currency(amount: float, from_currency: str, to_currency: str) -> str:
    """Convert an amount between two ISO currency codes."""
    return f"{amount * fx_rate(from_currency, to_currency):.2f} {to_currency}"

`create_agent` compiles a LangGraph graph that loops between the model and the
tools until the model answers without requesting another tool call. (It lives
in `langchain.agents` since LangGraph v1 — the older
`langgraph.prebuilt.create_react_agent` is deprecated.)

In [6]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

agent = create_agent(
    model=ChatOpenAI(model="gpt-4o-mini", temperature=0),
    tools=[get_weather, convert_currency],
)

## 2. Expose the agent as a scenario callable

A scenario's `outputs` callable must take `inputs` and return the value to
store in the trace — here, the agent's final message. The trajectory lives in
the graph's message history, so we extract the tool calls into a module-level
list at the same time.

Recording the trajectory in a module-level list is enough because scenarios and
suites run serially: each check reads the trajectory of the interaction that
just ran.

In [7]:
TOOL_CALLS: list[dict] = []


def run_agent(inputs: str) -> str:
    """Invoke the agent, record its tool calls, return its final answer."""
    state = agent.invoke({"messages": [{"role": "user", "content": inputs}]})

    TOOL_CALLS.clear()
    for message in state["messages"]:
        for call in getattr(message, "tool_calls", None) or []:
            TOOL_CALLS.append({"name": call["name"], "args": call["args"]})

    return state["messages"][-1].content

In [8]:
print(run_agent("What's the weather in Paris?"))
print(TOOL_CALLS)

The current weather in Paris is 18°C with light rain.
[{'name': 'get_weather', 'args': {'city': 'Paris'}}]


## 3. Check the final answer

Nothing agent-specific yet — this is the same `Scenario` you built in earlier
tutorials, with `run_agent` as the output callable.

In [9]:
from giskard.checks import Scenario, StringMatching

weather_answer = (
    Scenario("weather_answer")
    .interact(
        inputs="What's the weather in Paris?",
        outputs=lambda inputs: run_agent(inputs),
    )
    .check(
        StringMatching(
            name="mentions_rain",
            keyword="rain",
            text_key="trace.last.outputs",
        )
    )
)

result = await weather_answer.run()
result.print_report()

──────────────────────────────────────────────────── ✅ PASSED ────────────────────────────────────────────────────
mentions_rain   PASS    
────────────────────────────────────────────────────── Trace ──────────────────────────────────────────────────────
────────────────────────────────────────────────── Interaction 1 ──────────────────────────────────────────────────
Inputs: "What's the weather in Paris?"
Outputs: 'The current weather in Paris is 18°C with light rain.'
────────────────────────────────────────── 1 step in 1263ms | runs: 1/1 ───────────────────────────────────────────

## 4. Check the trajectory

The answer being right is not the same as the agent getting there correctly.
These checks read `TOOL_CALLS` instead of the trace, so they assert on *how*
the agent answered: that the right tool was chosen, with the right arguments,
and exactly once — a common agent failure is calling the same tool in a loop.

In [10]:
from giskard.checks import FnCheck

currency_trajectory = (
    Scenario("currency_trajectory")
    .interact(
        inputs="Convert 100 EUR to USD.",
        outputs=lambda inputs: run_agent(inputs),
    )
    .check(
        FnCheck(
            fn=lambda trace: [c["name"] for c in TOOL_CALLS] == ["convert_currency"],
            name="calls_convert_currency_once",
            failure_message="Expected exactly one convert_currency call",
        )
    )
    .check(
        FnCheck(
            fn=lambda trace: TOOL_CALLS[0]["args"]["from_currency"] == "EUR"
            and TOOL_CALLS[0]["args"]["to_currency"] == "USD",
            name="correct_currency_pair",
            failure_message="Agent converted the wrong currency pair",
        )
    )
)

result = await currency_trajectory.run()
result.print_report()
print(TOOL_CALLS)

──────────────────────────────────────────────────── ✅ PASSED ────────────────────────────────────────────────────
calls_convert_currency_once     PASS    
correct_currency_pair   PASS    
────────────────────────────────────────────────────── Trace ──────────────────────────────────────────────────────
────────────────────────────────────────────────── Interaction 1 ──────────────────────────────────────────────────
Inputs: 'Convert 100 EUR to USD.'
Outputs: '100 EUR is equivalent to 109.00 USD.'
────────────────────────────────────────── 1 step in 1337ms | runs: 1/1 ───────────────────────────────────────────

[{'name': 'convert_currency', 'args': {'amount': 100, 'from_currency': 'EUR', 'to_currency': 'USD'}}]


The mirror-image failure is an agent that reaches for a tool it does not need.
Assert the empty trajectory for questions the model can answer on its own:

In [11]:
no_tool_needed = (
    Scenario("no_tool_needed")
    .interact(
        inputs="Who won the 1998 football World Cup?",
        outputs=lambda inputs: run_agent(inputs),
    )
    .check(
        FnCheck(
            fn=lambda trace: TOOL_CALLS == [],
            name="no_tool_called",
            failure_message="Agent called a tool for a general-knowledge question",
        )
    )
)

result = await no_tool_needed.run()
result.print_report()

──────────────────────────────────────────────────── ✅ PASSED ────────────────────────────────────────────────────
no_tool_called  PASS    
────────────────────────────────────────────────────── Trace ──────────────────────────────────────────────────────
────────────────────────────────────────────────── Interaction 1 ──────────────────────────────────────────────────
Inputs: 'Who won the 1998 football World Cup?'
Outputs: "France won the 1998 FIFA World Cup. They hosted the tournament and defeated Brazil 3-0 in the final, 
which took place at the Stade de France in Saint-Denis, a suburb of Paris. This victory marked France's first World
Cup title."
────────────────────────────────────────── 1 step in 1077ms | runs: 1/1 ───────────────────────────────────────────

## 5. Spy on what the tool actually received

`TOOL_CALLS` tells you what the *model* asked for. `WithSpy` tells you what the
code *underneath* the tool received — useful when the tool transforms arguments
before hitting a database or an API.

`target` is the Python import path `mock.patch` will replace for the duration
of the interaction. Use `.add_interaction()` rather than `.interact()` when
passing a `WithSpy`.

In [12]:
from giskard.checks import Interact, WithSpy

SPY_TARGET = "__main__.weather_api"

spied_weather = WithSpy(
    interaction_generator=Interact(
        inputs="What's the weather in Berlin?",
        outputs=lambda inputs: run_agent(inputs),
    ),
    target=SPY_TARGET,
)

result = await Scenario("weather_api_args").add_interaction(spied_weather).run()

spy_data = result.final_trace.last.metadata.get(SPY_TARGET)
print(spy_data)

{'call_args_list': [call('Berlin')], 'call_count': 1, 'call_args': call('Berlin'), 'mock_calls': [call('Berlin'), call().__str__()]}


One thing to keep in mind: while the spy is active, `weather_api` is a
`MagicMock`, so the tool returns a placeholder and the agent's answer is
meaningless. That is expected — a spied interaction tests the call, not the answer.
Keep output checks in a separate, unspied scenario.

In [13]:
assert spy_data is not None, "No spy data — check the target import path"
assert spy_data["call_count"] == 1, "weather_api should be called exactly once"
assert spy_data["call_args"].args[0] == "Berlin", "Wrong city reached weather_api"

print(f"weather_api called once with {spy_data['call_args'].args[0]!r}")

weather_api called once with 'Berlin'


## 6. Judge the answer quality

Trajectory checks are exact; answer quality is not. Add an `LLMJudge` for the
part a string match cannot express. Register a generator once, and every
`LLMJudge` in the process uses it.

In [14]:
from giskard.checks import LLMJudge, set_default_generator
from giskard.agents.generators import Generator

set_default_generator(Generator(model="openai/gpt-4o-mini"))

answer_quality = (
    Scenario("answer_quality")
    .interact(
        inputs="What's the weather in Paris?",
        outputs=lambda inputs: run_agent(inputs),
    )
    .check(
        LLMJudge(
            name="answers_the_question",
            prompt="""
            Evaluate whether the assistant answered the user's question with
            concrete weather information rather than deflecting.

            User: {{ trace.last.inputs }}
            Assistant: {{ trace.last.outputs }}

            Return 'passed: true' if a temperature or condition is stated.
            """,
        )
    )
)

## 7. Run everything as a suite

Group the scenarios so one `await` covers answer, trajectory, and quality.

In [15]:
from giskard.checks import Suite

suite = (
    Suite(name="langgraph_agent_suite")
    .append(weather_answer)
    .append(currency_trajectory)
    .append(no_tool_needed)
    .append(answer_quality)
)

result = await suite.run()
result.print_report()

────────────────────────────────────────────────── Suite Results ──────────────────────────────────────────────────
....

───────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Summary: 4 total, 4 passed | Pass Rate: 100.0% | Total Duration: 6549ms

## The three layers

| Layer      | What it catches                              | Tool                    |
| ---------- | -------------------------------------------- | ----------------------- |
| Answer     | Wrong or missing information in the reply     | `StringMatching`, `LLMJudge` |
| Trajectory | Wrong tool, wrong arguments, redundant calls  | `FnCheck` over recorded tool calls |
| Internals  | Wrong values reaching the code under the tool | `WithSpy`               |

An agent that passes only the answer layer can be right by accident. The
trajectory and internals layers are what turn a passing test into evidence.

## See also

- [Spy on Internal Calls](/oss/checks/how-to/spy-on-calls) — `WithSpy` in
  depth, including multi-turn resets
- [Custom Checks](/oss/checks/how-to/custom-checks) — turn the trajectory
  assertions above into reusable `Check` subclasses
- [Checks reference](/oss/checks/reference/checks) — every built-in check